# RAG 튜토리얼 노트북

이 노트북은 RAG(Retrieval-Augmented Generation)의 전체 흐름을 단계별로 직접 실행해보는 학습용 파일입니다.

각 셀을 순서대로 실행하면서 중간 결과를 확인하고,  
이해한 내용을 바탕으로 `{이름}_exp01.py` 에 본인만의 방식으로 구현해보세요.

---

## 전체 흐름
```
PDF 파일
  → [1단계] 청킹: 문서를 작은 조각으로 분할
  → [2단계] 임베딩: 텍스트를 숫자 벡터로 변환
  → [3단계] 벡터 DB 저장
  → [4단계] 검색: 질문과 가장 유사한 청크 찾기
  → [5단계] LLM 답변 생성
  → [6단계] 평가
```

## 0. 환경 설정

In [ ]:
# 필요한 패키지 설치 (최초 1회)
# !pip install pymupdf chromadb sentence-transformers openai python-dotenv

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

# 프로젝트 루트 기준으로 .env 로드
load_dotenv(Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
print("API 키 로드:", "OK" if OPENAI_API_KEY else "없음 (.env 확인 필요)")

---
## 1단계. PDF 청킹

PDF를 읽어서 작은 텍스트 조각(청크)으로 나눕니다.  
청킹 방식에 따라 검색 품질이 크게 달라집니다.

**실험 포인트**: 청크 크기, 분할 기준(문단/문장/TOC), 오버랩 여부

In [ ]:
import fitz  # PyMuPDF

# ↓ 본인이 가진 PDF 경로로 바꾸세요
PDF_PATH = "../data/lg/aircon/SQ09GK1WEN.pdf"

doc = fitz.open(PDF_PATH)
print(f"총 페이지 수: {len(doc)}")
print(f"\n--- 1페이지 텍스트 미리보기 ---")
print(doc[0].get_text()[:500])

In [ ]:
# TOC(목차) 확인
toc = doc.get_toc()
print(f"TOC 항목 수: {len(toc)}")
print("\n--- TOC 구조 (처음 20개) ---")
for level, title, page in toc[:20]:
    print(f"{'  ' * (level-1)}[{level}] {title} (p.{page})")

In [ ]:
# 간단한 청킹 예시: 고정 크기로 분할
def simple_chunk(doc, chunk_size=500, overlap=50):
    """전체 텍스트를 고정 크기로 분할하는 가장 기본적인 방법."""
    full_text = ""
    for page in doc:
        full_text += page.get_text()
    
    chunks = []
    start = 0
    while start < len(full_text):
        end = min(start + chunk_size, len(full_text))
        chunks.append({
            "text": full_text[start:end],
            "start": start,
        })
        start += chunk_size - overlap
    return chunks

chunks = simple_chunk(doc, chunk_size=500, overlap=50)
print(f"생성된 청크 수: {len(chunks)}")
print(f"\n--- 첫 번째 청크 ---")
print(chunks[0]["text"])

In [ ]:
# 청크 크기 분포 확인
sizes = [len(c["text"]) for c in chunks]
print(f"최소: {min(sizes)}자")
print(f"최대: {max(sizes)}자")
print(f"평균: {sum(sizes)/len(sizes):.0f}자")

# 실험: chunk_size와 overlap을 바꿔보면 결과가 어떻게 달라지는지 확인해보세요

---
## 2단계. 임베딩

텍스트를 숫자 벡터로 변환합니다.  
의미가 비슷한 문장은 벡터 공간에서 가까운 위치에 놓입니다.

**실험 포인트**: 임베딩 모델 선택 (로컬 bge-m3 vs OpenAI)

In [ ]:
# 방법 A: OpenAI 임베딩 (빠름, 유료)
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def embed_openai(texts: list[str]) -> list[list[float]]:
    resp = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )
    return [item.embedding for item in resp.data]

# 테스트
sample_texts = ["에어컨 필터 청소 방법", "세탁기 탈수 오류"]
embeddings = embed_openai(sample_texts)
print(f"벡터 차원: {len(embeddings[0])}")
print(f"첫 번째 벡터 앞 5개: {embeddings[0][:5]}")

In [ ]:
# 의미적 유사도 확인
import numpy as np

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

texts = [
    "에어컨 필터 청소 방법",
    "에어컨 필터 세척하는 법",  # 의미적으로 유사
    "냉장고 온도 설정",          # 의미적으로 다름
]
vecs = embed_openai(texts)

print(f"'에어컨 필터 청소' vs '에어컨 필터 세척': {cosine_similarity(vecs[0], vecs[1]):.4f}")
print(f"'에어컨 필터 청소' vs '냉장고 온도 설정': {cosine_similarity(vecs[0], vecs[2]):.4f}")
print("→ 유사한 의미일수록 1에 가깝습니다")

---
## 3단계. 벡터 DB 저장

임베딩 벡터를 ChromaDB에 저장합니다.  
메타데이터(제품명, 섹션 등)를 함께 저장하면 나중에 필터링이 가능합니다.

In [ ]:
import chromadb

# 튜토리얼용 임시 DB (메모리)
client_db = chromadb.Client()
collection = client_db.create_collection("tutorial")

# 청크 5개만 저장해보기
sample_chunks = chunks[:5]
texts_to_embed = [c["text"] for c in sample_chunks]
vectors = embed_openai(texts_to_embed)

collection.add(
    ids=[f"chunk_{i}" for i in range(len(sample_chunks))],
    embeddings=vectors,
    documents=texts_to_embed,
    metadatas=[{"chunk_index": i, "product_model": "SQ09GK1WEN"} for i in range(len(sample_chunks))],
)

print(f"저장된 청크 수: {collection.count()}")

---
## 4단계. 검색 (Retrieval)

사용자 질문을 임베딩해서, 가장 유사한 청크를 찾습니다.  
distance 값이 낮을수록 더 유사한 문서입니다.

**실험 포인트**: top_k 수, 필터 조건, 중복 제거 방식

In [ ]:
query = "필터 청소 방법"

[query_vector] = embed_openai([query])

results = collection.query(
    query_embeddings=[query_vector],
    n_results=3,
)

print(f"질문: {query}")
print(f"\n--- 검색 결과 (top 3) ---")
for i, (doc, dist, meta) in enumerate(zip(
    results["documents"][0],
    results["distances"][0],
    results["metadatas"][0],
)):
    print(f"\n[{i+1}] distance={dist:.4f} | chunk_index={meta['chunk_index']}")
    print(doc[:150])

In [ ]:
# 여러 질문으로 검색 결과 비교해보기
test_queries = [
    "에어컨 필터 청소",
    "filter cleaning",     # 영어로도 검색되는지
    "소음이 나요",          # 전혀 다른 주제
]

for q in test_queries:
    [vec] = embed_openai([q])
    res = collection.query(query_embeddings=[vec], n_results=1)
    dist = res["distances"][0][0]
    print(f"질문: {q!r:20} → distance={dist:.4f}")

---
## 5단계. LLM 답변 생성

검색된 문서를 컨텍스트로 LLM에게 전달해서 답변을 생성합니다.  
**RAG의 핵심**: 검색된 문서 내용만 근거로 답하게 합니다.

**실험 포인트**: 시스템 프롬프트, 컨텍스트 구성 방식, 모델 선택

In [ ]:
def generate_answer(query: str, context_chunks: list[str]) -> str:
    context = "\n".join(f"- {c}" for c in context_chunks)
    
    system_prompt = (
        "너는 가전제품 사용법과 문제 해결을 도와주는 어시스턴트야. "
        "반드시 한국어로만 답해. "
        "아래 검색된 문서 내용만 근거로 답변해. 문서에 없는 내용은 지어내지 마."
    )
    
    user_message = f"[검색된 문서]\n{context}\n\n[사용자 질문]\n{query}"
    
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message},
        ],
    )
    return resp.choices[0].message.content


query = "필터 청소 방법"
[vec] = embed_openai([query])
res = collection.query(query_embeddings=[vec], n_results=3)
context_chunks = res["documents"][0]

answer = generate_answer(query, context_chunks)
print(f"질문: {query}")
print(f"\n--- 답변 ---")
print(answer)

In [ ]:
# 실험: 시스템 프롬프트를 바꿔보면 답변이 어떻게 달라지는지 확인해보세요
# 예) 더 친절하게, 더 간결하게, 단계별로 설명하도록 등

---
## 6단계. 평가

내 RAG가 얼마나 잘 작동하는지 공통 질문 셋으로 측정합니다.

결과 JSON을 저장하면 GitHub Actions가 자동으로 팀 전체 비교 보고서를 만들어줍니다.

In [ ]:
# 공통 평가 질문
COMMON_QUESTIONS = [
    ("에어컨 UE 오류가 뭐야?",              "UE",   "에러코드 단순 조회"),
    ("에어컨 필터 청소 방법 알려줘",         "필터", "일반 사용법"),
    ("세탁기 탈수가 너무 시끄러워",          "탈수", "증상 기반"),
    ("냉장고 온도를 어떻게 설정해?",         "온도", "설정/조작"),
    ("UE 오류랑 필터 청소 방법 같이 알려줘", "필터", "복합 질문"),
]

print("평가 질문 목록:")
for i, (q, kw, desc) in enumerate(COMMON_QUESTIONS, 1):
    print(f"  {i}. [{desc}] {q}")

In [ ]:
import time, json

def run_eval_simple(questions, embed_fn, answer_fn, top_k=3):
    results = []
    for question, expected_keyword, description in questions:
        start = time.time()
        [vec] = embed_fn([question])
        res = collection.query(query_embeddings=[vec], n_results=top_k)
        context = res["documents"][0]
        answer = answer_fn(question, context)
        elapsed = time.time() - start
        
        all_text = answer + " ".join(context)
        keyword_hit = expected_keyword in all_text
        distances = res["distances"][0]
        
        hit_mark = "O" if keyword_hit else "X"
        print(f"[{hit_mark}] {description}: {question}")
        print(f"    elapsed={elapsed:.1f}s | avg_dist={sum(distances)/len(distances):.4f}")
        
        results.append({
            "question": question,
            "description": description,
            "keyword_hit": keyword_hit,
            "elapsed_sec": round(elapsed, 2),
            "avg_distance": round(sum(distances)/len(distances), 4),
            "answer": answer,
        })
    return results

# 주의: 실제 실행하면 OpenAI API 비용이 발생합니다
# eval_results = run_eval_simple(COMMON_QUESTIONS, embed_openai, generate_answer)

In [ ]:
# 결과 저장 (GitHub Actions가 읽어서 비교 보고서 생성)
# ↓ 본인 이름과 실험 번호로 바꾸세요
EXPERIMENT_NAME = "홍길동_exp01"

# eval_results가 있을 때 실행
# report = {
#     "experiment_name": EXPERIMENT_NAME,
#     "notes": "고정 크기 청킹 + OpenAI 임베딩 베이스라인",
#     "summary": {
#         "keyword_hit_rate": sum(r["keyword_hit"] for r in eval_results) / len(eval_results),
#         "avg_distance": sum(r["avg_distance"] for r in eval_results) / len(eval_results),
#         "avg_elapsed_sec": sum(r["elapsed_sec"] for r in eval_results) / len(eval_results),
#     },
#     "results": eval_results,
# }
#
# os.makedirs("results", exist_ok=True)
# with open(f"results/{EXPERIMENT_NAME}.json", "w", encoding="utf-8") as f:
#     json.dump(report, f, ensure_ascii=False, indent=2)
# print(f"저장 완료: results/{EXPERIMENT_NAME}.json")

---
## 다음 단계: 본인 실험 파일 작성

이 노트북에서 각 단계를 이해했으면, 이제 `{이름}_exp01.py`를 만들어보세요.

```bash
cp template_exp.py 홍길동_exp01.py
```

**`my_answer()` 함수에서 실험할 수 있는 것들:**

| 실험 아이디어 | 변경 포인트 |
|---|---|
| 청크 크기를 바꿔보기 | `chunk_size=300` vs `chunk_size=800` |
| 검색 결과 개수 변경 | `top_k=3` vs `top_k=5` |
| 프롬프트 개선 | system_prompt 내용 수정 |
| 임베딩 모델 변경 | OpenAI vs bge-m3 |
| 중복 제거 방식 | 섹션 기준 dedup 로직 수정 |